# Aiscern — Image AI Detector Fine-Tuning
**Platform:** Kaggle (T4 GPU)  
**Target:** `saghi776/aiscern-image-detector`  
**Base model:** `google/vit-base-patch16-224-in21k`  
**Runtime:** ~45 minutes  
**Expected accuracy:** >88% on CIFAKE test set

### Before running:
1. Set Accelerator → **GPU T4 x2** in notebook settings
2. Add secret `HF_TOKEN` in Add-ons → Secrets (needs **write** permission)
3. Create `saghi776/aiscern-image-detector` repo on huggingface.co/new

In [ ]:
# CELL 1 — Install dependencies
!pip install -q transformers==4.40.0 datasets accelerate huggingface_hub \
    pillow torch torchvision scikit-learn evaluate peft

In [ ]:
# CELL 2 — Authenticate HuggingFace
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets   = UserSecretsClient()
HF_TOKEN  = secrets.get_secret('HF_TOKEN')
login(token=HF_TOKEN, add_to_git_credential=False)
print('✅ Authenticated with HuggingFace')

In [ ]:
# CELL 3 — Configuration
import torch

MODEL_ID    = 'google/vit-base-patch16-224-in21k'
REPO_ID     = 'saghi776/aiscern-image-detector'
BATCH_SIZE  = 32
EPOCHS      = 3
LR          = 2e-5
SEED        = 42
MAX_SAMPLES = 25000   # 12.5K AI + 12.5K real — fits in Kaggle time budget
IMG_SIZE    = 224
OUTPUT_DIR  = '/kaggle/working/aiscern-image-detector'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# CELL 4 — Load CIFAKE dataset (primary — clean, balanced, pre-labeled)
# CIFAKE: 60K AI (Stable Diffusion) + 60K Real (CIFAR-10 photographs)
# Labels: 'REAL' = 0, 'FAKE' = 1 (we remap to REAL=0, AI_GENERATED=1)
from datasets import load_dataset, concatenate_datasets
import numpy as np

print('Loading CIFAKE...')
cifake_train = load_dataset(
    'jlbaker361/cifake-real-and-ai-generated-small-images',
    split='train',
    token=HF_TOKEN,
)
cifake_test = load_dataset(
    'jlbaker361/cifake-real-and-ai-generated-small-images',
    split='test',
    token=HF_TOKEN,
)

print(f'CIFAKE train: {len(cifake_train):,} samples')
print(f'CIFAKE test:  {len(cifake_test):,} samples')
print(f'Columns: {cifake_train.column_names}')
print(f'Label distribution: {cifake_train.features}')

In [ ]:
# CELL 5 — Normalise labels and balance dataset
# CIFAKE label column may be 'label' (0=REAL, 1=FAKE) or 'Label' (string)

def normalise_label(example):
    # Handle both int and string labels
    lbl = example.get('label', example.get('Label', 0))
    if isinstance(lbl, str):
        lbl = 1 if lbl.upper() in ('FAKE', 'AI', 'AI_GENERATED', '1') else 0
    return {'label': int(lbl)}

cifake_train = cifake_train.map(normalise_label)
cifake_test  = cifake_test.map(normalise_label)

# Balance: equal AI and real samples
half = MAX_SAMPLES // 2
real_train = cifake_train.filter(lambda x: x['label'] == 0).shuffle(SEED).select(range(min(half, cifake_train.filter(lambda x: x['label'] == 0).num_rows)))
fake_train = cifake_train.filter(lambda x: x['label'] == 1).shuffle(SEED).select(range(min(half, cifake_train.filter(lambda x: x['label'] == 1).num_rows)))
balanced   = concatenate_datasets([real_train, fake_train]).shuffle(SEED)

# Split: 90% train, 10% val
split      = balanced.train_test_split(test_size=0.1, seed=SEED)
train_ds   = split['train']
val_ds     = split['test']
test_ds    = cifake_test  # official test split

print(f'Train: {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,}')
real_count = sum(1 for x in train_ds if x['label'] == 0)
print(f'Train balance — Real: {real_count:,}  AI: {len(train_ds)-real_count:,}')

In [ ]:
# CELL 6 — Image preprocessing with ViT feature extractor
from transformers import ViTImageProcessor
from PIL import Image as PILImage
import torch

processor = ViTImageProcessor.from_pretrained(MODEL_ID)

def preprocess(examples):
    images = []
    for img in examples['image']:
        if not isinstance(img, PILImage.Image):
            img = PILImage.fromarray(img)
        images.append(img.convert('RGB').resize((IMG_SIZE, IMG_SIZE)))
    inputs = processor(images=images, return_tensors='pt')
    inputs['labels'] = torch.tensor(examples['label'])
    return inputs

print('Preprocessing train...')
train_ds = train_ds.map(preprocess, batched=True, batch_size=256,
                         remove_columns=[c for c in train_ds.column_names if c not in ('pixel_values','labels')],
                         desc='Train')
print('Preprocessing val...')
val_ds   = val_ds.map(preprocess, batched=True, batch_size=256,
                       remove_columns=[c for c in val_ds.column_names if c not in ('pixel_values','labels')],
                       desc='Val')
print('Preprocessing test...')
test_ds  = test_ds.map(preprocess, batched=True, batch_size=256,
                        remove_columns=[c for c in test_ds.column_names if c not in ('pixel_values','labels')],
                        desc='Test')

train_ds.set_format('torch')
val_ds.set_format('torch')
test_ds.set_format('torch')
print('✅ Preprocessing complete')

In [ ]:
# CELL 7 — Load ViT for binary classification
from transformers import ViTForImageClassification

model = ViTForImageClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    id2label={0: 'REAL', 1: 'AI_GENERATED'},
    label2id={'REAL': 0, 'AI_GENERATED': 1},
    ignore_mismatched_sizes=True,
)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params:     {total_params/1e6:.1f}M')
print(f'Trainable params: {trainable_params/1e6:.1f}M')

In [ ]:
# CELL 8 — Training arguments
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    warmup_ratio=0.1,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=50,
    fp16=True,
    dataloader_num_workers=4,
    push_to_hub=True,
    hub_model_id=REPO_ID,
    hub_token=HF_TOKEN,
    hub_strategy='every_save',
    report_to='none',
    seed=SEED,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1':       f1_score(labels, preds, average='binary'),
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print('✅ Trainer ready')

In [ ]:
# CELL 9 — Train
print(f'Training on {len(train_ds):,} samples for {EPOCHS} epochs...')
print(f'Batch size: {BATCH_SIZE}  LR: {LR}  FP16: True')
trainer.train()

In [ ]:
# CELL 10 — Evaluate on held-out test set
print('Evaluating on CIFAKE test set...')
test_results = trainer.evaluate(test_ds)

print(f"\n{'='*40}")
print(f"TEST ACCURACY: {test_results['eval_accuracy']:.4f}  ({test_results['eval_accuracy']*100:.1f}%)")
print(f"TEST F1:       {test_results['eval_f1']:.4f}")
print(f"{'='*40}")

if test_results['eval_accuracy'] >= 0.88:
    print('✅ Target accuracy >88% achieved!')
else:
    print(f"⚠️  Below 88% target — consider more epochs or more data")

In [ ]:
# CELL 11 — Push to HuggingFace Hub
commit_msg = f"ViT fine-tuned for AI image detection — accuracy={test_results['eval_accuracy']:.3f} f1={test_results['eval_f1']:.3f}"
trainer.push_to_hub(commit_message=commit_msg)
processor.push_to_hub(REPO_ID, token=HF_TOKEN)
print(f'\n✅ Model pushed to: https://huggingface.co/{REPO_ID}')

In [ ]:
# CELL 12 — Verify inference API output format
# hf-analyze.ts uses: /ai|fake|generated|artificial/i → matches 'AI_GENERATED'
# and: /real|authentic|genuine/i → matches 'REAL'
# Output MUST be [{label, score}, {label, score}] from image-classification pipeline

from transformers import pipeline
import time

print('Verifying model output format...')
pipe = pipeline('image-classification', model=REPO_ID, token=HF_TOKEN, device=0 if device=='cuda' else -1)

# Test with a sample from the test set
sample_idx = 0
sample_img  = test_ds[sample_idx]
# Reconstruct PIL image from pixel_values tensor for pipeline
from torchvision.transforms.functional import to_pil_image
pil_img = to_pil_image(sample_img['pixel_values'])

result = pipe(pil_img)
print(f'\nInference API output:')
print(result)
print(f'\nExpected format: [{{"label": "AI_GENERATED", "score": X}}, {{"label": "REAL", "score": Y}}]')

# Validate format
labels_in_output = [r['label'] for r in result]
assert 'AI_GENERATED' in labels_in_output, f'ERROR: AI_GENERATED label missing! Got: {labels_in_output}'
assert 'REAL' in labels_in_output, f'ERROR: REAL label missing! Got: {labels_in_output}'
print('\n✅ Output format correct — compatible with hf-analyze.ts')